In [1]:
import json,os
import re
import time
import subprocess
from concurrent.futures import ThreadPoolExecutor, as_completed
from openai import OpenAI
from tqdm import tqdm      
import base64
import requests
from time import sleep
from io import BytesIO 
import random
import shutil
from PIL import Image
import math

def copy_file(source_path, destination_path):
    """
    Copy a file from source path to destination path
    
    Parameters:
    source_path (str): Path to the source file
    destination_path (str): Path where the file should be copied to
    
    Returns:
    bool: True if copy was successful, False otherwise
    """
    try:
        shutil.copy2(source_path, destination_path)
        print(f"Successfully copied file from {source_path} to {destination_path}")
        return True
    except Exception as e:
        print(f"Error copying file: {e}")
        return False

def read_jsonl(file_path):
    """
    读取 JSONL 文件并返回列表
    
    :param file_path: JSONL 文件路径
    :return: 包含所有 JSON 对象的列表
    """
    data = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                data.append(json.loads(line))
            except json.JSONDecodeError as e:
                print(f"Error decoding JSON: {e}, Line {line}")
    return data

def extract_between_answer(text):
    start = text.find("<answer>") + len("<answer>")
    end = text.find("</answer>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def extract_between_think(text):
    start = text.find("<think>") + len("<think>")
    end = text.find("</think>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""


def extract_between_tool_call(text):
    start = text.find("<tool_call>") + len("<tool_call>")
    end = text.find("</tool_call>")
    if start != -1 and end != -1:
        return text[start:end].strip()
    return ""

def split_tool_call(text: str) -> tuple[str, str, str]:
    """
    将文本拆分成三段：
    before  : <tool_call> 之前的字符串
    middle  : <tool_call> 与 </tool_call> 之间的字符串（去掉首尾空白）
    after   : </tool_call> 之后的字符串

    如果标记不存在，则 middle 为空串，before 为原始文本，after 为空串。
    """
    open_tag, close_tag = "<tool_call>", "</tool_call>"

    start = text.find(open_tag)
    end   = text.find(close_tag, start + len(open_tag))  # 从 open_tag 之后再找，避免嵌套误匹配

    # 没找到成对标记，直接返回
    if start == -1 or end == -1:
        return text, "", ""

    before = text[:start]
    middle = text[start + len(open_tag) : end].strip()
    after  = text[end + len(close_tag) :]

    return before, middle, after

# data preparation

In [9]:
interactive_data_1 = read_jsonl("/home/aiqihang.aqh/Appagent/data/interactive_data.jsonl")
interactive_data_2 = read_jsonl("/home/aiqihang.aqh/Appagent/data/【iTAG】交互原因交互内容标注_0530_v2_UTF__20250628102014.jsonl")
image_folder = "/home/aiqihang.aqh/Appagent/annotation/images"


pos_sft_data = []


system_prompt = '''You are a mobile GUI agent. You are given a task with the current screenshot. You need to perform the next action to complete the task.

You are provided with function signatures within <tools></tools> XML tags:

<tools>
{{"type": "function", "function": {{"name\_for\_human": "mobile\_use", "name": 
"mobile\_use", "description": "Use a touchscreen to interact with a mobile device."}}}}
</tools>

For each function call, return a json object with function name and arguments within <tool\_call></tool\_call> XML tags:

<tool\_call>
{{"name": <function-name>, "arguments": <args-json-object>}}
</tool\_call>'''


for data in interactive_data_1:

    answer = json.loads(extract_between_answer(data['solution']))

    new_answer = {
        "name": "mobile_use",
        "arguments": answer
    }

    think = extract_between_think(data['solution'])

    instruction = data['instruction']


    user_prompt = f'''分析任务和当前图片，给出下一步操作。

    在标签<think> </think>内输出思考过程。
    在标签<tool\_call> </tool\_call>内输出最终答案。

    用户任务: {instruction}'''

    gt_response = f'''<think>
{think}
</think>
<tool_call>
{json.dumps(new_answer, ensure_ascii=False)}
</tool_call>'''


    sft_data = {
        "conversations": [
        {
            "from": "system",
            "value": system_prompt,
        },
        {
            "from": "human",
            "value": "<image>"+user_prompt
        },
        {
            "from": "gpt",
            "value": gt_response
        }
        ],
        "images": [os.path.join(image_folder, data['image'].split("/")[-1])]
    }


    pos_sft_data.append(sft_data)

for data in interactive_data_2:

    action = {
        "action": "call_user",
        "content": data['交互内容']
    }

    new_answer = {
        "name": "mobile_use",
        "arguments": action
    }

    think = data['交互原因']

    instruction = data['instruction']


    user_prompt = f'''分析任务和当前图片，给出下一步操作。

在标签<think> </think>内输出思考过程。
在标签<tool\_call> </tool\_call>内输出最终答案。

用户任务: {instruction}'''

    gt_response = f'''<think>
{think}
</think>
<tool_call>
{json.dumps(new_answer, ensure_ascii=False)}
</tool_call>'''


    sft_data = {
        "conversations": [
        {
            "from": "system",
            "value": system_prompt,
        },
        {
            "from": "human",
            "value": "<image>"+user_prompt
        },
        {
            "from": "gpt",
            "value": gt_response
        }
        ],
        "images": [os.path.join(image_folder, data['image_name'])]
    }


    pos_sft_data.append(sft_data)

Error decoding JSON: Expecting property name enclosed in double quotes: line 1 column 602 (char 601), Line {"任务ID": "1928340219616731136", "子任务包ID": "1928340350440554496", "数据集ID": "1928335789022789632", "数据ID": "1928302095137263630", "think": "用户似乎对学习编程感兴趣，但可能不确定从哪个方向或资源开始。图像表明这可能是针对儿童或初学者的。", "image_name": "eb03d058-db49-4642-a6b0-c2055a84323b.png", "instruction": "I decided to start learning programming to build a solid foundation for a future career in technology.", "type": "call_user", "content": "您是否在寻找适合初学者的编程资源或课程？如果您在编程方面有特定的目标或兴趣领域，请告诉我，以便我能更好地帮助您。", "screenshot": "https://yuqiaochuang-agent.oss-cn-beijing.aliyuncs.com/agent/eb03d058-db49-4642-a6b0-c2055a84323b.png", "介入原因": "[\"意图确认\"]", x "The user instruction is \"Learning to program as a foundation for a career,\" but it does not specify operation details. It's an intent confirmation and needs the user to confirm. The current screenshot shows a Xiaohongshu note details page, and it's necessary to confirm the user's intent

In [7]:
len(pos_sft_data)

974

In [4]:
with open("/home/aiqihang.aqh/Appagent/data/sft_pos.jsonl", "w", encoding='utf-8') as file:
    for data in pos_sft_data:
        json.dump(data, file, ensure_ascii=False)
        file.write("\n")

# coordinate

In [2]:
input_file_path = "/home/aiqihang.aqh/Appagent/data/sft_general.jsonl"
image_folder = "/home/aiqihang.aqh/GUI-R1/new_images"


all_data = read_jsonl(input_file_path)
data = all_data[10]

data

{'conversations': [{'from': 'system',
   'value': 'You are a mobile GUI agent. You are given a task and your action history, with the current screenshot and the previous state preceding the last action. You need to perform the next action to complete the task.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n\n<tools>\n{{"type": "function", "function": {{"name\\_for\\_human": "mobile\\_use", "name": \n"mobile\\_use", "description": "Use a touchscreen to interact with a mobile device."}}}}\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool\\_call></tool\\_call> XML tags:\n\n<tool\\_call>\n{{"name": <function-name>, "arguments": <args-json-object>}}\n</tool\\_call>\n'},
  {'from': 'human',
   'value': '<image><image>分析任务和历史动作，给出下一步操作。\n\n在标签<think> </think>内输出思考过程。\n在标签<tool\\_call> </tool\\_call>内输出最终答案。\n\n用户任务: 打开小红书，搜索“点淘“， 筛选里选择排序依据”最新“，找到第一个笔记，关注笔记的作者\n\n历史动作: 打开小红书应用。点击搜索图标，以便进行搜索操作。输入“点淘”到搜索框中。'},


In [3]:
new_data = []

for idx, data in enumerate(all_data):

    try:

        data['images'] = [os.path.join(image_folder, image) for image in data['images']]

        img = Image.open(data['images'][-1])   # 得到 PngImageFile 对象
        image_w, image_h = img.size   

        pixels = image_w*image_h
        max_pixels = 1280*28*28

        if pixels > max_pixels:

            
            scale_factor = math.sqrt(max_pixels/pixels)
            before, action, after =  split_tool_call(data['conversations'][2]['value'])
            action = json.loads(action)
            # print(action)

            type_ = action['arguments']['action']

            if type_ == 'click':
                x,y = action['arguments']['coordinate']
                new_x, new_y = int(x*scale_factor), int(y*scale_factor)
                action['arguments']['coordinate'] = [new_x, new_y]

            if type_ == 'swipe':
                x1,y1 = action['arguments']['coordinate1']
                new_x1, new_y1 = int(x1*scale_factor), int(y1*scale_factor)
                action['arguments']['coordinate1'] = [new_x1, new_y1]
                x2,y2 = action['arguments']['coordinate2']
                new_x2, new_y2 = int(x2*scale_factor), int(y2*scale_factor)
                action['arguments']['coordinate2'] = [new_x2, new_y2]

            data['conversations'][2]['value'] = f"{before}<tool_call>\n{json.dumps(action)}\n</tool_call>"

        new_data.append(data)

    except:
        continue

len(new_data)

1052

In [2]:
new_data = read_jsonl("/home/aiqihang.aqh/Appagent/data/interactive_sft.jsonl")
file_path = "/home/aiqihang.aqh/Appagent/data/interactive_sft.json"
with open(file_path, "w", encoding="utf-8") as f:
    json.dump(new_data, f, ensure_ascii=False, indent=2)